# 00. PyTorch Warmup | PyTorch 核心基础热身: 张量、前反向传播与 Embedding (Warmup)

In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

### Part 1: 张量维度变换与 `einops`
> 举个典型的例子：将形状为 `[batch, heads, seq_len, head_dim]` 的多头张量合并为 `[batch, seq_len, hidden_dim]`：
> - **原生实现**：`x.permute(0, 2, 1, 3).reshape(batch, seq_len, -1)` 
> - **`einops` 实现**：`rearrange(x, 'b h s d -> b s (h d)')` 

### Part 2: 嵌入层 (Embedding Layer) 的本质

>文本是离散的（Token IDs，如 `[10, 42, 99]`）。神经网络只能处理连续的稠密向量（Dense Vectors）。
>**Embedding 层的本质：** 就是一个大规模的查表（Lookup Table）。给定一个 ID 列表，它直接把对应的行向量抽出来拼在一起。
>它在数学上等价于：把离散的 ID 转换成 One-hot 向量，然后去乘以一个全连接层（Linear）。


In [2]:
def embedding_warmup(input_ids: torch.Tensor, vocab_size: int, hidden_dim: int):
    """
    演示 Embedding 查表的过程，并用纯 Tensor 索引模拟它。
    
    Args:
        input_ids: 形状 [batch_size, seq_len]，包含整数类型的 Token IDs
    """
    # ==========================================
    # TODO 2.1: 实例化一个官方的 nn.Embedding，并用其进行前向传播
    # ==========================================
    emb_layer = nn.Embedding(vocab_size,hidden_dim)
    emb_layer.weight.data.normal_(0, 0.1)  # 就地（in-place）用正态分布填充，均值=0，标准差=0.1
    out_official = emb_layer(input_ids)
    
    # ==========================================
    # TODO 2.2: 使用纯 PyTorch 张量索引 (Advanced Indexing)，不使用 nn.Embedding，
    # 达到和上面官方 API 完全一模一样的输出。
    # 提示: Embedding 的本质是查表，思考如何用索引从权重矩阵中提取向量
    # ==========================================
    out_manual = emb_layer.weight[input_ids]
    
    return out_official, out_manual
    pass

### Part 3: 前向传播与反向传播 (Forward & Backward)

In [3]:
class LinearReLUFunction(torch.autograd.Function):
    """
    实现一个包含 Linear + ReLU 的算子，并推导其反向传播的梯度。
    公式: y = relu(x @ W^T + b)
    """
    @staticmethod
    def forward(ctx, x, weight, bias):
        # ==========================================
        # TODO 3.1: 实现前向传播
        # 1. 使用 F.linear 计算线性变换
        # 2. 使用 F.relu 计算激活
        # 3. 计算并保存 mask，用于反向传播时判断哪些位置需要传递梯度
        # 4. 保存必要的张量供反向传播使用
        # ==========================================
        z = F.linear(x, weight, bias)
        y = F.relu(z)
        mask = (z > 0).float()
        ctx.save_for_backward(x, weight, mask)
        return y
        pass

    @staticmethod
    def backward(ctx, grad_output):
        """
        接收从上一层传回来的梯度 (grad_output)，形状同 y。
        返回对当前层三个输入 (x, weight, bias) 的梯度。
        """
        x, weight, mask = ctx.saved_tensors
        
        # ==========================================
        # TODO 3.2: 反传过 ReLU
        # 提示: ReLU 的导数在正值处为 1，负值处为 0
        # ==========================================
        grad_z = grad_output * mask
        
        # ==========================================
        # TODO 3.3: 反传过 Linear
        # 提示: 利用矩阵求导的链式法则，分别计算对 x, weight, bias 的梯度
        # 注意矩阵维度的匹配和转置操作
        # ==========================================
        grad_x = grad_z @ weight
        grad_weight = grad_z.T @ x
        grad_bias = grad_z.sum(dim=0)
        
        return grad_x, grad_weight, grad_bias
        pass

# 01. RMSNorm Tutorial | 均方根层归一化 (RMSNorm)

### LayerNorm 核心公式与张量维度

给定输入向量 $x \in \mathbb{R}^d$，LayerNorm 的输出 $y$ 为：

1. **计算均值与方差：**
   $$ \mu = \frac{1}{d} \sum_{i=1}^d x_i $$
   $$ \sigma^2 = \frac{1}{d} \sum_{i=1}^d (x_i - \mu)^2 + \epsilon $$
   其中 $\epsilon$ 是为了防止除以 0 的极小值（如 `1e-5`）。

2. **归一化：**
   $$ \hat{x} = \frac{x - \mu}{\sqrt{\sigma^2}} $$

3. **缩放并平移：**
   $$ y = \hat{x} \odot \gamma + \beta $$
   其中 $\gamma \in \mathbb{R}^d$ 是可学习的缩放权重（Weight），$\beta \in \mathbb{R}^d$ 是可学习的偏移参数（Bias）。

**与 RMSNorm 的对比：**
- LayerNorm 包含**均值中心化**（减去 $\mu$）和**方差缩放**两步；RMSNorm 只做 RMS 缩放，无中心化。
- LayerNorm 有**两个**可学习参数（$\gamma$ 和 $\beta$），RMSNorm 仅有 $\gamma$。
- LayerNorm 是原始 Transformer 的标准归一化，RMSNorm 是其变体，计算量更小。

### RMSNorm 核心公式与张量维度

给定输入向量 $x \in \mathbb{R}^d$，RMSNorm 的输出 $y$ 为：

1. **计算均方根 (RMS)：**
   $$ \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon} $$
   其中 $\epsilon$ 是为了防止除以 0 的极小值（如 `1e-6`）。

2. **归一化并缩放 (Scale)：**
   $$ y = \frac{x}{\text{RMS}(x)} \odot \gamma $$
   其中 $\gamma \in \mathbb{R}^d$ 是可学习的权重参数（Weight）。**RMSNorm 没有偏置项 (Bias)**。


### 数值溢出 (Numerical Overflow)

> **工程经验：为什么要强制转换精度？**
> 现代大模型训练与推理几乎都会使用混合精度 (AMP) 或半精度格式 (`FP16`) 以节省显存。但在执行平方和均值操作前，通常需要显式地将其转换为 `float32` 计算。待归一化计算完毕后，再将结果转换回原有精度。

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # ==========================================
        # TODO 1: 定义可学习参数 weight，并初始化为全 1
        # 形状: [hidden_size]
        # 提示: 使用 nn.Parameter 包装张量使其可学习
        # ==========================================
        self.weight = nn.Parameter(torch.ones(hidden_size))
        pass
        

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 2: 实现 RMSNorm 核心计算逻辑
        # 提示: 
        # 1. 为防止 FP16 溢出，需要在高精度下计算
        # 2. 计算输入的均方值（平方后求均值），注意保持维度以便广播
        # 3. 使用均方根的倒数进行归一化，torch.rsqrt 比 1/sqrt 更快
        # 4. 返回归一化后的结果（保持高精度，便于后续操作）
        # ==========================================
        x_fp32 = x if x.dtype == torch.float32 else x.float()
        variance = x_fp32.pow(2).mean(dim=-1, keepdim=True)
        return x_fp32 * torch.rsqrt(variance + self.eps)
        pass


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 3: 组合归一化与权重缩放
        # 提示: 调用 _norm 进行归一化，乘以可学习的 weight，最后转回输入精度
        # ==========================================
        weight = self.weight.to(x.dtype)
        return (weight * self._norm(x)).to(x.dtype)
        pass

# 02. SwiGLU Activation | 激活函数与门控机制 (SwiGLU Activation)

### ReLU（Rectified Linear Unit）

**核心公式：**
$$ \text{ReLU}(x) = \max(0, x) $$

---

### SiLU（Sigmoid Linear Unit）/ Swish

> **注：** SiLU 也叫 Swish，由 Google 提出（$\beta=1$ 时），是 Swish 的特例。

**核心公式：**
$$ \text{SiLU}(x) = x \cdot \sigma(x) = \frac{x}{1 + e^{-x}} $$
其中 $\sigma(x)$ 是 Sigmoid 函数。

**函数特性：**
- 当 $x \to +\infty$ 时，SiLU$(x) \to x$（近似线性）；当 $x \to -\infty$ 时，SiLU$(x) \to 0$。
- **导数：** $\text{SiLU}'(x) = \sigma(x) + x \cdot \sigma(x) \cdot (1 - \sigma(x))$。

### GLU 与 SwiGLU

#### 1. GLU（Gated Linear Unit）

**核心思想：**  
传统 MLP 的前馈计算为 $W_{down}(\sigma(W_{up}x))$，即“激活 → 线性变换”。  
GLU 引入**门控机制**，将输入分成两条路径：

- **门控路径（Gate）**：经过激活函数，控制信息流动的“开关”。
- **线性路径（Value）**：保持线性变换，作为待传递的内容。

两条路径逐元素相乘（Hadamard Product），再经过输出投影。

**公式：**
$$ \text{GLU}(x, W_1, W_2, W_3) = (xW_1 \odot \sigma(xW_2)) W_3 $$

---

#### 2. SwiGLU（Swish-Gated Linear Unit）

**核心改进：**  
将 GLU 中的激活函数 $\sigma$ 替换为 **Swish**（即 $x \cdot \text{Sigmoid}(\beta x)$）。

**公式：**
$$ \text{SwiGLU}(x, W_1, W_2, W_3) = (xW_1 \odot \text{Swish}(xW_2)) W_3 $$

其中 $\text{Swish}(z) = z \cdot \sigma(z) = \frac{z}{1 + e^{-z}}$（PyTorch 中 $\beta=1$ 时等同于 `SiLU`）。

#### 3. 与传统 MLP 的结构对比

| 结构 | 公式 | 参数量 |
|------|------|--------|
| 标准 MLP（ReLU） | $W_3 \cdot \text{ReLU}(W_1 x)$ | $2dh$ |
| GLU | $(W_1 x \odot \sigma(W_2 x)) W_3$ | $3dh$（多一个门控矩阵） |
| SwiGLU | $(W_1 x \odot \text{Swish}(W_2 x)) W_3$ | $3dh$（同 GLU） |

### 参数量对齐

**典型的面试问题：**
> “在 GPT-2 中，隐藏层维度通常是输入维度 $d$ 的 4 倍（即 $4d$）。但在使用 SwiGLU 的 LLaMA 中，为什么隐藏层维度变成了 $\frac{8}{3}d$ 并向上取整？”

**推导过程：**
1. **标准 MLP 参数量**：
   输入为 $d$，隐藏层为 $h$。
   有两个投影矩阵（升维 $d \to h$，降维 $h \to d$）。
   总参数量 = $2 \cdot (d \times h)$。
   当 $h = 4d$ 时，总参数量 = $2 \cdot 4d^2 = \mathbf{8d^2}$。

2. **SwiGLU MLP 参数量**：
   输入为 $d$，隐藏层为 $h_{swiglu}$。
   因为有**门控机制**，升维阶段需要**两个**投影矩阵（$W_{gate}$ 和 $W_{up}$，均是 $d \to h_{swiglu}$）。
   降维阶段需要**一个**矩阵（$W_{down}$，是 $h_{swiglu} \to d$）。
   总参数量 = $3 \cdot (d \times h_{swiglu})$。

3. **对齐参数量**：
   为了使得 SwiGLU 的计算开销（参数量）与原始模型完全相同：
   $3 \cdot d \cdot h_{swiglu} = 8d^2$
   解得：$h_{swiglu} = \mathbf{\frac{8}{3}d}$
   
这正是 LLaMA 源码中对中间层维度进行 `int(8 * hidden_size / 3)` 计算的根本原因。

### 工业级实现框架与性能陷阱 (Memory Bound)

**性能陷阱 1：张量并行 (TP) 与内存对齐**
在真实的 LLaMA 源码中，除了按 $8/3$ 计算出隐藏层维度，还需要将其向上取整对齐到一个 `multiple_of`（通常是 256）的倍数。
这不仅是为了让单卡 Tensor Core（通常要求 8-byte 或 32-byte 对齐）跑得更快，更是因为大模型训练会使用**张量并行 (Tensor Parallelism)**。如果隐藏层维度不能被 GPU 数量整除（例如 $TP=8$ 时，256 的倍数分给 8 张卡，每张卡至少能分到 32 维），在切分权重矩阵时就会发生严重的报错。

**性能陷阱 2：访存瓶颈 (Memory Bound) 与矩阵融合**
在最朴素的代码实现中，开发者会分别定义并执行 `gate_proj(x)` 和 `up_proj(x)`。
由于这两个线性层**共享完全相同的输入张量 $x$**，分开计算会导致巨大的输入 $x$ 被 GPU 从全局显存 (HBM) 中读取两次。
> **工业界解法 (Matrix Fusion)**：
> 在 vLLM、Megatron 等主流框架中，标准的做法是将 $W_{gate}$ 和 $W_{up}$ 这两个形状为 `[hidden_size, intermediate_size]` 的权重矩阵，在初始化时拼接成一个**巨大的融合矩阵 `gate_up_proj`**，其形状为 `[hidden_size, 2 * intermediate_size]`。
> 
> 在前向传播时，输入 $x$ 只需要被读取一次进行一次大规模矩阵乘法。得到的结果再通过 `torch.chunk(2, dim=-1)` 切分为两半，分别作为 gate 和 up 块。这极大地缓解了内存带宽瓶颈。

In [5]:
def calculate_intermediate_size(hidden_size: int, multiple_of: int = 256):
    """
    计算 LLaMA 风格的 SwiGLU 隐藏层维度
    
    规则：
    1. 取 hidden_size 的 8/3
    2. 为了硬件对齐（如 Tensor Core），通常要求是 multiple_of 的倍数。
       因此将结果除以 multiple_of，向上取整后再乘以 multiple_of。
    """
    # ==========================================
    # TODO 1: 计算理论隐藏层大小 (8/3 * hidden_size)
    # 提示: 注意使用整数除法
    # ==========================================
    intermediate_size = int(hidden_size * 8 / 3)
    
    # ==========================================
    # TODO 2: 向 multiple_of 对齐 (向上取整)
    # 提示: 思考如何利用整除的特性实现向上取整
    # ==========================================
    aligned_size = ((intermediate_size + multiple_of - 1) // multiple_of) * multiple_of
    
    return aligned_size
    pass

class SwiGLU_MLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # ==========================================
        # TODO 3: 定义工业级 SwiGLU 的投影矩阵
        # ==========================================
        self.gate_up_proj = nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 4: 组装工业级 SwiGLU 前向传播
        # ==========================================
        gate_up = self.gate_up_proj(x)
        gate, up = torch.chunk(gate_up, 2, dim=-1)
        return self.down_proj(F.silu(gate) * up)
        pass

# 03. RoPE Tutorial | 旋转位置编码 (RoPE)

### RoPE（Rotary Position Embedding）详解

#### 1. 核心思想

**痛点：** 绝对位置编码（正弦波或可学习）让模型难以泛化到更长序列，且无法显式建模 Token 间的相对距离。

**RoPE 的解决方案：** 通过旋转矩阵将位置信息注入 Q 和 K，使内积结果自动包含相对距离 $(m-n)$，且无需额外参数。

---

#### 2. 2D 旋转基础

对于 2D 向量 $(x_1, x_2)$，位置 $m$ 的旋转矩阵为：

$$ R_{m\theta} = \begin{bmatrix} \cos(m\theta) & -\sin(m\theta) \\ \sin(m\theta) & \cos(m\theta) \end{bmatrix} $$

旋转后：
$$ x_{rotated} = R_{m\theta} \cdot x $$

**核心性质：** 两个旋转后的向量点积只依赖角度差 $(m-n)\theta$，与绝对位置无关。

---

#### 3. 扩展到高维

将 $d$ 维向量分成 $d/2$ 组，每组 2 维独立旋转，使用不同频率：

$$ \theta_j = 10000^{-2j/d}, \quad j = 0, 1, \dots, d/2-1 $$

第 $j$ 组旋转角度为 $m \cdot \theta_j$。

**高频（大 $\theta$）→ 捕捉短距离；低频（小 $\theta$）→ 捕捉长距离。**

---

#### 4. 复数乘法实现

**为什么用复数？** 因为 $e^{i\theta} = \cos\theta + i\sin\theta$，复数乘法等价于 2D 旋转。

**实现步骤：**
1. 将向量 $x \in \mathbb{R}^d$ 前 $d/2$ 维作实部，后 $d/2$ 维作虚部：$x_{complex} = x_{[:d/2]} + i \cdot x_{[d/2:]}$
2. 预计算旋转因子 $e^{im\theta_j} = \cos(m\theta_j) + i\sin(m\theta_j)$
3. 复数乘法：$x_{rotated\_complex} = x_{complex} \cdot e^{im\Theta}$
4. 还原为实数：实部放回前半，虚部放回后半

In [6]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    计算复数指数频率张量 (cis = cos + i * sin)
    """
    # ==========================================
    # TODO 1: 用极坐标生成复数张量 (提示: torch.polar)
    # ==========================================
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)                                                                                                            
    return freqs_cis   
    

def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor):
    ndim = x.ndim
    shape = [d if i == 1 or i == ndim - 1 else 1 for i, d in enumerate(x.shape)]
    return freqs_cis.view(*shape)

def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    将旋转位置编码应用到 Query 和 Key 上
    """
    # ==========================================
    # TODO 2: 将 xq, xk 从实数张量转为复数张量
    # 提示: 
    # ==========================================
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
                                                                                                                                                                         
    freqs_cis = reshape_for_broadcast(freqs_cis, xq_)
    
    # ==========================================
    # TODO 3: 进行复数乘法，并转回实数张量
    # 提示: 
    # ==========================================
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
                                                                                                                                                              
                 
    return xq_out.type_as(xq), xk_out.type_as(xk)      


# 04. Attention MHA GQA | 注意力机制与键值缓存 (MHA / GQA / MQA)

> 读取巨量的 KV Cache 会面临严重的**显存容量瓶颈**和**内存带宽瓶颈 (Memory-bound)**，导致推理极慢。

> * **MHA (Multi-Head Attention)**
> * **MQA (Multi-Query Attention)**
> * **GQA (Grouped-Query Attention)**
> *  **MLA(Multi-head Latent Attention)**

下面是一个GQA的实现

In [7]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    将 KV 头复制 n_rep 次，以匹配 Query 头的数量 (GQA/MQA 需要)
    """
    batch, num_kv_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_kv_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_kv_heads * n_rep, slen, head_dim)

class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_dim: int, num_heads: int, num_kv_heads: int = None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else num_heads
        
        self.num_queries_per_kv = self.num_heads // self.num_kv_heads
        self.head_dim = hidden_dim // num_heads
        
        # 定义投影矩阵
        self.q_proj = nn.Linear(hidden_dim, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, hidden_dim, bias=False)

    def forward(
        self, 
        x: torch.Tensor, 
        attention_mask: torch.Tensor = None, 
        kv_cache: tuple[torch.Tensor, torch.Tensor] = None
    ):
        batch_size, seq_len, _ = x.shape
        
        # 1. 线性投影
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        
        # ==========================================
        # TODO 1: Reshape xq, xk, xv 以适配多头注意力计算
        # ==========================================
        xq = xq.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        xk = xk.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        xv = xv.view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        # xq = ???
        # xk = ???
        # xv = ???
        
        
        # ==========================================
        # TODO 2: 处理 KV Cache
        # ==========================================
        if kv_cache is not None:
            k_cache, v_cache = kv_cache
            xk = torch.cat([k_cache, xk], dim=2)
            xv = torch.cat([v_cache, xv], dim=2)
            
        new_kv_cache = (xk, xv)
        
        # 通过 repeat_kv 把 GQA 的 KV 头数扩充到和 Query 数量一致
        xk = repeat_kv(xk, self.num_queries_per_kv)
        xv = repeat_kv(xv, self.num_queries_per_kv)
        
        # ==========================================
        # TODO 3: 计算注意力分数 (Scaled Dot-Product)
        # ==========================================
        scores = torch.matmul(xq, xk.transpose(2, 3)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            scores = scores + attention_mask
            
        probs = torch.nn.functional.softmax(scores, dim=-1)
        output = torch.matmul(probs, xv)
        
        # ==========================================
        # TODO 4: 恢复形状并输出
        # [B, H, S, D] -> [B, S, H*D]
        # ==========================================
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        
        return self.o_proj(output), new_kv_cache
        pass
